In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

spark = SparkSession.builder\
        .master('local[*]')\
        .appName('test')\
        .getOrCreate()

26/02/22 00:15:49 WARN Utils: Your hostname, codespaces-473e1b resolves to a loopback address: 127.0.0.1; using 10.0.1.125 instead (on interface eth0)
26/02/22 00:15:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/02/22 00:15:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
import pandas as pd

pd_df = pd.read_csv('./notebooks/head.csv')

In [3]:
pd_df.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [5]:
spark.createDataFrame(pd_df).schema

StructType(List(StructField(hvfhs_license_num,StringType,true),StructField(dispatching_base_num,StringType,true),StructField(pickup_datetime,StringType,true),StructField(dropoff_datetime,StringType,true),StructField(PULocationID,LongType,true),StructField(DOLocationID,LongType,true),StructField(SR_Flag,DoubleType,true)))

In [10]:
schema = types.StructType([
    types.StructField("hvfhs_license_num",types.StringType(),True),
    types.StructField("dispatching_base_num",types.StringType(),True),
    types.StructField("pickup_datetime",types.StringType(),True),
    types.StructField("dropoff_datetime",types.StringType(),True),
    types.StructField("PULocationID",types.IntegerType(),True),
    types.StructField("DOLocationID",types.IntegerType(),True),
    types.StructField("SR_Flag",types.DoubleType(),True)
])


In [11]:
df_trips = spark.read\
            .schema(schema)\
            .option('header', 'true')\
            .csv("./notebooks/fhvhv_tripdata_2021-01.csv")
df_trips.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: double (nullable = true)



In [13]:
df_trips = df_trips.repartition(24)

In [14]:
df_trips.write.parquet('./data/fhvhv/2021/01/')

In [15]:
df_pq = spark.read.parquet('./data/fhvhv/2021/01/')

In [18]:
df_pq.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: double (nullable = true)

